#### 1. 누적 매출액에 대한 회귀 분석 -> 전체 영화에 대한 회귀 분석

In [181]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# 그래프상에서 한글을 나타내기 위해 korean_mat 설치
!pip install koreanize-matplotlib
import koreanize_matplotlib

%config InlineBackend.figure_format = 'retina'

# 데이터 불러오기
df = pd.read_csv('data/movie_finished_toFDA_V9.csv')
df.head()


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


#### 영화별로 기본 모형에 따라 어떤 것이 누적 매출액에 어떤 식으로 영향을 주는 지 살펴본다

* 기본 모형_1: 누적 매출액 = 국적 + 장르 + 등급 + 감독 + 배우 : 영화의 내적 요인
* 기본 모형_2: 누적 매출액 = 국적 + 장르 + 등급 + 감독 + 배우 + 스크린수 + 경쟁 영화 수  : 영화의 내적 요인 + 영화의 외부요인
* 기본 모형_3: 누적 매출액 = 국적 + 장르 + 등급 + 감독 + 배우 + 스크린수 + 경쟁 영화 수 + 포털 평점 + 소셜미디어 : 영화의 내적 요인 + 영화의 외부요인 + 입소문 요인

#### 변수 설명
* 1. 누적 매출액: 2주간의 영화 매출액
* ----- 독립변수 ------
* ----영화의 내적 요인 --------
* 2. 국적: 영화가 어느 국적인가 -> 한국 영화인가? 혹은 외국 영화인가?
* 3. 장르: 영화의 장르 정보
* 4. 등급: 영화 -> 12세이하, 12세 이상 15세 ...
* 5. 감독: 감독 파워를 사용
* 6. 배우: 배우 파워를 사용
* ----영화의 외부 요인 --------
* 7. 첫째주 스크린 점유율 -> 개봉사 파워? 혹은 
* 8. 개봉 경쟁 영화 수
* ----영화의 입소문 요인-----
* 9. 실관람객 평점
* 10. 소셜미디어 정보

In [182]:
# 국적 칼럼
col_national = ['nationality']

# 장르 정보: 
col_genre= ['사극', 'SF', '어드벤처', '액션', '범죄', '판타지', '가족', '미스터리', '스릴러', '뮤지컬',
       '코미디', '공포(호러)', '애니메이션', '드라마', '서부극(웨스턴)', '전쟁', '멜로/로맨스', '기타',
       '다큐멘터리', '공연', '성인물(에로)']

# 관람 가능 연령 타입
col_watch_grade = ['watch_grade']

# 영화 감독 파워
col_direc_power = ['director_power']

# 영화 배우 파워
col_act_power = ['actor_power']

# 영화 상영 정보
col_screen = ['스크린점유율']

# 영화 평점 정보
col_score = ['RV_score_total']

# 영화의 경쟁 요인


# 소셜미디어 정보 -> 채영이가 만든 입소문 지수를 사용?

# movie_title
col_movie_title = ['movie_title']

# 종속변수
col_cum_money = ['누적_매출액']

#### 영화의 종속변수 -> 누적 매출액

In [183]:
# 1. 영화별 누적 매출액을 계산
sum_viewer = df.groupby('movie_title')['매출액'].sum()

# 2. 원본 데이터프레임에 누적 관객수를 매핑하여 새로운 컬럼으로 추가
df['누적_매출액'] = df['movie_title'].map(sum_viewer)

#### 1. 전체 영화에 대해서 누적 매출액에 대한 첫번째 모델 검증
* 기본 모형_1: 누적 매출액 = 국적 + 장르 + 등급 + 감독 + 배우 : 영화의 내적 요인


In [184]:
# 전체 영화에 대한 첫번째 모델
df_model_1 = df[col_national+col_genre+col_watch_grade+col_direc_power+col_act_power+col_movie_title+col_cum_money].copy()
df_model_1

#### 분석을 위한 변수 전처리
* 국적 -> 만약 국적에 한국이 있다면 한국영화 -> 더미처리
* 장르 -> 더미 처리되어 있음
* 등급 -> 더미 처리
* 감독 파워 -> log 처리(영화마다 값의 차이가 너무 커서 log처리함)
* 배우 파워 -> log 처리(영화마다 값의 차이가 너무 커서 log처리함)


* 데이터 전처리를 위한 함수 정의

In [185]:
def create_dummies(df, columns, drop_first=True): # 더미 변수 생성을 위한 함수
    return pd.get_dummies(df, columns=columns, drop_first=drop_first)

def convert_data_types(df): # True,False -> 0,1로 변환
    df.update(df.select_dtypes(include=['bool']).astype(int))
    return df

def apply_log_transformation(df, columns): # 로그 변환
    for column in columns:
        df[column] = np.log1p(df[column])
    return df

def compute_daily_averages(df, daily_columns, key='movie_title'): # 일별 데이터 집계
    daily_averages = df.groupby(key)[daily_columns].mean().reset_index()
    df = df.drop(columns=daily_columns).merge(daily_averages, on=key, how='left')
    return df.groupby(key).mean().reset_index()

In [186]:
# 국적에 대해 한국인지 아닌지로 처리
# '한국영화'라는 값이 있으면 1로, 그렇지 않으면 0으로 변경 -> 1이면 한국영화
df_model_1['nationality'] = df_model_1['nationality'].apply(lambda x: '한국영화' if '한국' in x else '외국영화')

# 더미 변수 생성
df_model_1 = create_dummies(df_model_1, ['nationality', 'watch_grade'])

# 데이터 타입 변환
df_model_1 = convert_data_types(df_model_1)

# 로그 처리
log_columns = ['director_power', 'actor_power','누적_매출액']
df_model_1 = apply_log_transformation(df_model_1, log_columns)

/var/folders/mz/tk5z94ds1qd9c1jb2p8qys9m0000gn/T/ipykernel_76225/1031698659.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1 1 1 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.update(df.select_dtypes(include=['bool']).astype(int))
/var/folders/mz/tk5z94ds1qd9c1jb2p8qys9m0000gn/T/ipykernel_76225/1031698659.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1 1 1 ... 1 1 1]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.update(df.select_dtypes(include=['bool']).astype(int))
/var/folders/mz/tk5z94ds1qd9c1jb2p8qys9m0000gn/T/ipykernel_76225/1031698659.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly

In [187]:
df_model_1 = df_model_1.groupby('movie_title').mean()

#### 1. 첫번째 모델로 회귀 분석

In [188]:
import statsmodels.api as sm


# 독립변수(X)와 종속변수(y) 설정
X = df_model_1.drop('누적_매출액', axis=1)
y = df_model_1['누적_매출액']

# X에 상수항 추가 (절편 포함)
X = sm.add_constant(X)

# 회귀 모델 생성 및 적합
model = sm.OLS(y, X).fit()

model.summary()

####  --> 영화의 내적 특성들로 회귀 모델을 돌릴 경우에 R_score가 너무 낮게 나오고, 일부 칼럼의 상관계수가 도메인 지식과 반대로 나옴(watch_grade)

#### 2. 두번째 모델
* 기본 모형_2: 누적 매출액 = 국적 + 장르 + 등급 + 감독 + 배우 + 스크린수 + 경쟁 영화 수  : 영화의 내적 요인 + 영화의 외부요인

* 기본 모형_1과 달리, 스크린수와 경쟁영화수라는 칼럼이 추가되었다. -> 이는 영화의 외부요인이다
* 스크린수 -> 첫째주 스크린 점유율을 사용하였다. eda 결과 상영사와 배급사는 영화의 흥행 여부를 주단위로 예측해 영화의 스크린 점유율이 달라진다. 따라서 전체 스크린 점유율을 사용하는 것은 옳지 않다고 생각했다. 따라서 첫째주 스크린 점유율 평균을 사용하였다. 
* 영화의 경쟁 영화 수 -> 개봉일 기준 7일전 영화 수 + 5일전 영화 수 + 5일후 영화 수 + 7일후 영화 수 

In [189]:
# 영화마다 첫번째 일요일이 나올때까지 sum에 스크린 점유율을 합한다 -> 만약 영화의 개봉일이 목요일인 경우 sum + 목요일 스크린 점유율 + 금요일 스크린 점유율 + 토요일 스크린 점유율 + 일요일 스크린 점유율
# 일자를 정렬 순서로 설정
df[['movie_title','DoW','스크린점유율']]
# 요일 순서 지정
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['DoW'] = pd.Categorical(df['DoW'], categories=day_order, ordered=True)

# 결과 저장용 리스트
results = []

# 영화별로 그룹화
for movie, group in df.groupby('movie_title'):
    # 첫 번째 일요일의 인덱스 찾기
    first_sunday_idx = group[group['DoW'] == 'Sunday'].index.min()
    
    # 첫 번째 일요일까지 데이터 필터링
    filtered_group = group.loc[:first_sunday_idx]
    
    # 스크린 점유율 평균 계산
    avg_screen_share = filtered_group['스크린점유율'].mean()
    
    # 결과 저장
    results.append({'movie_title': movie, '1주차스크린점유율': avg_screen_share})

# 결과를 데이터프레임으로 변환
result_df = pd.DataFrame(results)
result_df[result_df['movie_title'] == '군함도']

# df와 병합
df= df.merge(result_df, on='movie_title', how='left')



In [190]:
# 영화의 경쟁영화 수 -> 개봉일 기준 7일전 영화 수 + 5일전 영화 수 + 5일후 영화 수 + 7일후 영화 수
df['2주동안의 경쟁영화수'] = df['7days_before'] + df['5days_before'] + df['Day_of_release'] + df['5days_after'] + df['7days_after']

In [191]:
# 새로운 칼럼
# 1주차 스크린 점유율
screen_col = ['1주차스크린점유율']

# 경쟁 영화 수
compete_col = ['2주동안의 경쟁영화수']

In [192]:
df_model_2 = df[col_national+col_genre+col_watch_grade+col_direc_power+col_act_power+screen_col+compete_col+col_movie_title+col_cum_money].copy()

* 모델을 돌리기 위해 데이터 전처리

In [193]:
# 국적에 대해 한국인지 아닌지로 처리
# '한국영화'라는 값이 있으면 1로, 그렇지 않으면 0으로 변경 -> 1이면 한국영화
df_model_2['nationality'] = df_model_2['nationality'].apply(lambda x: '한국영화' if '한국' in x else '외국영화')

# 더미 변수 생성
df_model_2 = create_dummies(df_model_2, ['nationality', 'watch_grade'])

# 데이터 타입 변환
df_model_2 = convert_data_types(df_model_2)

# 로그 처리
log_columns = ['director_power', 'actor_power','2주동안의 경쟁영화수','누적_매출액']
df_model_2 = apply_log_transformation(df_model_2, log_columns)

df_model_2 = df_model_2.groupby('movie_title').mean()

/var/folders/mz/tk5z94ds1qd9c1jb2p8qys9m0000gn/T/ipykernel_76225/1031698659.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1 1 1 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.update(df.select_dtypes(include=['bool']).astype(int))
/var/folders/mz/tk5z94ds1qd9c1jb2p8qys9m0000gn/T/ipykernel_76225/1031698659.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1 1 1 ... 1 1 1]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.update(df.select_dtypes(include=['bool']).astype(int))
/var/folders/mz/tk5z94ds1qd9c1jb2p8qys9m0000gn/T/ipykernel_76225/1031698659.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly

In [194]:
import statsmodels.api as sm


# 독립변수(X)와 종속변수(y) 설정
X = df_model_2.drop('누적_매출액', axis=1)
y = df_model_2['누적_매출액']

# X에 상수항 추가 (절편 포함)
X = sm.add_constant(X)

# 회귀 모델 생성 및 적합
model = sm.OLS(y, X).fit()

model.summary()

#### 기본 모형_3: 누적 매출액 = 국적 + 장르 + 등급 + 감독 + 배우 + 스크린수 + 경쟁 영화 수 + 포털 평점 + 소셜미디어 : 영화의 내적 요인 + 영화의 외부요인 + 입소문 요인

* 기존의 모형에서 입소문 요인을 추가하였다. -> 1주차 실관람객 평점

In [195]:
# 영화의 1주차_실관람객 평점을 입소문 요인으로서 추가하였다
# 영화마다 첫번째 일요일이 나올때까지 sum에 실관람객 평점을 합한다 -> 만약 영화의 개봉일이 목요일인 경우 sum + 목요일 스크린 점유율 + 금요일 스크린 점유율 + 토요일 스크린 점유율 + 일요일 스크린 점유율
# 일자를 정렬 순서로 설정
df[['movie_title','DoW','RV_Dailyscore']]
# 요일 순서 지정
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['DoW'] = pd.Categorical(df['DoW'], categories=day_order, ordered=True)

# 결과 저장용 리스트
results = []

# 영화별로 그룹화
for movie, group in df.groupby('movie_title'):
    # 첫 번째 일요일의 인덱스 찾기
    first_sunday_idx = group[group['DoW'] == 'Sunday'].index.min()
    
    # 첫 번째 일요일까지 데이터 필터링
    filtered_group = group.loc[:first_sunday_idx]
    
    # 실관람객 평균 계산
    avg_daily_score = filtered_group['RV_Dailyscore'].mean()
    
    # 결과 저장
    results.append({'movie_title': movie, '1주차실관람객평균평점': avg_daily_score})

# 결과를 데이터프레임으로 변환
result_df = pd.DataFrame(results)

# df와 병합
df= df.merge(result_df, on='movie_title', how='left')

* 1주차실관람객평균평점 -> nan값이 세개 존재 -> 관람객 평균 평점(NTZ_Dailyscore)으로 대체
* 나의 소녀시대,업그레이드,러브 유어셀프 인 서울

In [221]:
 df['1주차실관람객평균평점'].fillna(99999,inplace=True)
df[df['1주차실관람객평균평점']==99999][['NTZ_Dailyscore','movie_title']]

df[df['movie_title'] == '나의 소녀시대'][['NTZ_Dailyscore','DoW']] # 9.4
df[df['movie_title'] == '업그레이드'][['NTZ_Dailyscore','DoW']]# 8.8
df[df['movie_title'] == '러브 유어셀프 인 서울'][['NTZ_Dailyscore','DoW']] # 9.3

replace_dic = {'나의 소녀시대':9.4,'업그레이드':8.8,'러브 유어셀프 인 서울':9.3}




/var/folders/mz/tk5z94ds1qd9c1jb2p8qys9m0000gn/T/ipykernel_76225/3151583924.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['1주차실관람객평균평점'].fillna(99999,inplace=True)


* 입소문 요인 칼럼 -> 1주차실관람객평균평점
* 모델 3: 기본 모형_3: 누적 매출액 = 국적 + 장르 + 등급 + 감독 + 배우 + 스크린수 + 경쟁 영화 수 + 포털 평점 + 소셜미디어 : 영화의 내적 요인 + 영화의 외부요인 + 입소문 요인

In [196]:
score_col = ['1주차실관람객평균평점']

In [223]:
# null 처리: 실관람객 평점이 존재하지 않은 3개의 영화에 대해서는 네티즌 평균 평점(1주차)로 처리
# NaN 값을 영화별로 특정 값으로 채우기
df.loc[df['movie_title'] == '나의 소녀시대', '1주차실관람객평균평점'] = 9.4
df.loc[df['movie_title'] == '업그레이드', '1주차실관람객평균평점'] = 8.8
df.loc[df['movie_title'] == '러브 유어셀프 인 서울', '1주차실관람객평균평점'] = 9.3





In [224]:
df_model_3 = df[col_national+col_genre+col_watch_grade+col_direc_power+col_act_power+screen_col+compete_col+score_col+col_movie_title+col_cum_money].copy()

* 모델 생성을 위한 전처리

In [225]:
# 국적에 대해 한국인지 아닌지로 처리
# '한국영화'라는 값이 있으면 1로, 그렇지 않으면 0으로 변경 -> 1이면 한국영화
df_model_3['nationality'] = df_model_3['nationality'].apply(lambda x: '한국영화' if '한국' in x else '외국영화')

# 더미 변수 생성
df_model_3 = create_dummies(df_model_3, ['nationality', 'watch_grade'])

# 데이터 타입 변환
df_model_3 = convert_data_types(df_model_3)

# 로그 처리
log_columns = ['director_power', 'actor_power','2주동안의 경쟁영화수','1주차실관람객평균평점','누적_매출액']
df_model_3 = apply_log_transformation(df_model_3, log_columns)

df_model_3 = df_model_3.groupby('movie_title').mean()

/var/folders/mz/tk5z94ds1qd9c1jb2p8qys9m0000gn/T/ipykernel_76225/1031698659.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1 1 1 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.update(df.select_dtypes(include=['bool']).astype(int))
/var/folders/mz/tk5z94ds1qd9c1jb2p8qys9m0000gn/T/ipykernel_76225/1031698659.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1 1 1 ... 1 1 1]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.update(df.select_dtypes(include=['bool']).astype(int))
/var/folders/mz/tk5z94ds1qd9c1jb2p8qys9m0000gn/T/ipykernel_76225/1031698659.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly

In [ ]:
# 독립변수(X)와 종속변수(y) 설정
X = df_model_3.drop('누적_매출액', axis=1)
y = df_model_3['누적_매출액']

# X에 상수항 추가 (절편 포함)
X = sm.add_constant(X)

# 회귀 모델 생성 및 적합
model = sm.OLS(y, X).fit()

model.summary()

#### 후진 제거법을 사용하여 모델을 돌려보기 -> 독립 변수는 모델 3을 사용

In [227]:
import pandas as pd
import statsmodels.api as sm
import itertools
import time

# 변수 선택 과정을 수행하는 함수
def processSubset(X, y, feature_set):
    model = sm.OLS(y, sm.add_constant(X[list(feature_set)]))
    regression = model.fit()
    return {"model": regression, "AIC": regression.aic}

# 후진제거법
def backward(X, y, predictors):
    results = []
    for combo in itertools.combinations(predictors, len(predictors) - 1):
        results.append(processSubset(X, y, combo))
    models = pd.DataFrame(results)
    best_model = models.loc[models['AIC'].argmin()]
    return best_model

# 후진제거법을 사용한 전체 모델 선택
def backward_model(X, y):
    predictors = X.columns.tolist()
    Bmodels = pd.DataFrame(columns=["AIC", "model"], index = range(1, len(X.columns)))
    
    while len(predictors) > 1:
        current_model = backward(X, y, predictors)
        Bmodels.loc[len(predictors) - 1] = current_model
        predictors = current_model['model'].model.exog_names
        predictors = [p for p in predictors if p != 'const']  # 상수항 제외
        
    # 최적의 모델 추출
    best_model_index = Bmodels['AIC'].idxmin()
    best_model = Bmodels.loc[best_model_index]
    return best_model

In [228]:
# 데이터 준비
# 독립변수(X)와 종속변수(y) 설정
X = df_model_3.drop('누적_매출액', axis=1)
y = df_model_3['누적_매출액']

# 후진제거법을 이용한 최적 모델 탐색
final_model = backward_model(X, y)

final_model['model'].summary()

#### 종속변수를 수익성으로 하고, 모델 3의 변수를 사용하여 후진 제거법을 사용

* 수익성 칼럼 만들기

In [239]:
# 15번째 행 간격으로 데이터 필터링
filtered_df = df.iloc[::15]

# '수익성' 계산
filtered_df['수익성'] = filtered_df['누적_매출액'] / filtered_df['좌석수']

# 계산된 '수익성'을 원래 데이터프레임으로 매핑
df['수익성'] = None  # 초기화
df.loc[filtered_df.index, '수익성'] = filtered_df['수익성']

/var/folders/mz/tk5z94ds1qd9c1jb2p8qys9m0000gn/T/ipykernel_76225/2342551628.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['수익성'] = filtered_df['누적_매출액'] / filtered_df['좌석수']


In [240]:
# 'movie_title'별 첫 번째 '수익성' 값을 그룹의 모든 행에 적용
df['수익성'] = df.groupby('movie_title')['수익성'].transform('first')

# 결과 확인
df[['movie_title', '누적_매출액', '좌석수', '수익성']]

In [241]:
# 수익성
profit_col = ['수익성']

In [245]:
df_model_3 = pd.merge(df_model_3,df[['movie_title','수익성']],on='movie_title')

In [263]:
df_model_3 = df_model_3.groupby('movie_title').mean()
df_model_3['수익성'] = pd.to_numeric(df_model_3['수익성'], errors='coerce')
df_model_3['수익성']= np.log1p(df_model_3['수익성'])

movie_title
0.0MHz          9.045517
12 솔져스          9.443582
1987           10.437006
1급기밀            8.995740
23 아이덴티티       10.583162
                 ...    
혹성탈출: 종의 전쟁     9.843569
혼숨              9.831606
후드              8.860007
희생부활자           8.248096
히든 피겨스          9.571594
Name: 수익성, Length: 409, dtype: float64

In [266]:
# 데이터 준비
# 독립변수(X)와 종속변수(y) 설정
X = df_model_3.drop(['누적_매출액','수익성'], axis=1)
y = df_model_3['수익성']

# 후진제거법을 이용한 최적 모델 탐색
final_model = backward_model(X, y)

final_model['model'].summary()

#### 현재까지는 모든 수치형 변수들에 대해 로그 변환을 해왔디 -> but 변수마다 다른 스케일링을 사용한다면?

* 후진 제거법을 하기 위한 + 각각의 다른 스케일릴을 위한 df 만들기

In [280]:
df_model_back = df[col_national+col_genre+col_watch_grade+col_direc_power+col_act_power+screen_col+compete_col+score_col+col_movie_title+col_cum_money+profit_col].copy()

* 분포가 정규분포가 아닌 것들에 대해서만 log변환을 취한다.

In [281]:
log_columns = ['director_power', 'actor_power','누적_매출액','수익성']

df_model_back['수익성'] = pd.to_numeric(df_model_back['수익성'], errors='coerce')

df_model_back = apply_log_transformation(df_model_back, log_columns)

* 다른 변수들에 대한 변환

In [282]:
# 국적에 대해 한국인지 아닌지로 처리
# '한국영화'라는 값이 있으면 1로, 그렇지 않으면 0으로 변경 -> 1이면 한국영화
df_model_back['nationality'] = df_model_back['nationality'].apply(lambda x: '한국영화' if '한국' in x else '외국영화')

# 더미 변수 생성
df_model_back = create_dummies(df_model_back, ['nationality', 'watch_grade'])

# 데이터 타입 변환
df_model_back = convert_data_types(df_model_back)

df_model_back = df_model_back.groupby('movie_title').mean()

/var/folders/mz/tk5z94ds1qd9c1jb2p8qys9m0000gn/T/ipykernel_76225/1031698659.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1 1 1 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.update(df.select_dtypes(include=['bool']).astype(int))
/var/folders/mz/tk5z94ds1qd9c1jb2p8qys9m0000gn/T/ipykernel_76225/1031698659.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1 1 1 ... 1 1 1]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.update(df.select_dtypes(include=['bool']).astype(int))
/var/folders/mz/tk5z94ds1qd9c1jb2p8qys9m0000gn/T/ipykernel_76225/1031698659.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly

In [283]:
df_model_back

#### 1. 종속변수를 log(누적_매출액)로 하여 후진 제거법

In [284]:
# 데이터 준비
# 독립변수(X)와 종속변수(y) 설정
X = df_model_back.drop(['누적_매출액','수익성'], axis=1)
y = df_model_back['누적_매출액']

# 후진제거법을 이용한 최적 모델 탐색
final_model = backward_model(X, y)

final_model['model'].summary()

#### 2. 종속변수를 log(수익성)으로 하여 후진제거법

In [285]:
# 데이터 준비
# 독립변수(X)와 종속변수(y) 설정
X = df_model_back.drop(['누적_매출액','수익성'], axis=1)
y = df_model_back['수익성']

# 후진제거법을 이용한 최적 모델 탐색
final_model = backward_model(X, y)

final_model['model'].summary()

In [286]:
a = 'hi'